In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

df = pd.read_csv(r"/content/creditcard.csv")
print(df.head())
print("Dataset Shape:", df.shape)
print(df['Class'].value_counts())
print(df.isnull().sum().sum())
print(df.info())

X = df.drop('Class', axis=1)
y = df['Class']
print("Input Shape:", X.shape)
print("Output Shape:", y.shape)

df_cleaned = df.dropna()
X = df_cleaned.drop('Class', axis=1)
y = df_cleaned['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)

scaler = StandardScaler()
X_train = X_train.copy()
X_test = X_test.copy()
X_train[['Time', 'Amount']] = scaler.fit_transform(X_train[['Time', 'Amount']])
X_test[['Time', 'Amount']] = scaler.transform(X_test[['Time', 'Amount']])

smote = SMOTE(sampling_strategy=0.1, k_neighbors=1, random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)
print("After SMOTE:")
print(y_train.value_counts())

model = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, eval_metric='logloss', random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Model Training Completed")

y_pred = model.predict(X_test)
print("Classification Report")
print(classification_report(y_test, y_pred))
print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred))
print("ROC AUC Score:")
print(roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]))

def detect_fraud(transaction):
    start_time = time.perf_counter()
    # Convert Series to DataFrame matching model features
    trans_df = transaction.to_frame().T
    prediction = model.predict(trans_df)[0]
    probability = model.predict_proba(trans_df)[0][1]
    end_time = time.perf_counter()
    response_time = (end_time - start_time) * 1000

    if prediction == 1:
        result = "FRAUD DETECTED"
    else:
        result = "NORMAL TRANSACTION"

    print("--------------------------------")
    print("Transaction Amount:", transaction['Amount'])
    print("Prediction:", result)
    print("Fraud Probability:", round(probability * 100, 2), "%")
    print("Processing Time:", round(response_time, 3), "ms")
    print("--------------------------------")

import random
for i in range(10):
    transaction_number = random.randint(0, len(X_test) - 1)
    transaction = X_test.iloc[transaction_number]
    print(f"\nTransaction Number: {i + 1}")
    detect_fraud(transaction)
    time.sleep(1)

   Time        V1        V2        V3        V4        V5        V6        V7  \
0     0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1     0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2     1 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3     1 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4     2 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

        V26       V27       V28 